In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""           # Force CPU

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)




In [ ]:
# ============================================================
# 1. LOAD AND PARSE DATA
# ============================================================

sig_df = pd.read_csv('test_data_signal.csv', header=None)       # Signal file
bkg_df = pd.read_csv('test_data_background.csv', header=None)   # Background file

MAX_PARTICLES = 50             # Max particles per event
N_FEAT_PER_PARTICLE = 4        # pT, eta, phi, flag
OBJ_START = 8                  # Column where per-particle data begins



In [ ]:

def parse_data(df, max_particles=MAX_PARTICLES):
    """
    Parse raw CSV into structured arrays.
    
    Returns:
    --------
    event_features : (n_events, 7)
    particle_features : (n_events, max_particles, 4)
    particle_mask : (n_events, max_particles)
    """
    raw = df.values                                        # Convert to numpy
    n_events = len(df)
    
    # Event-level features: columns 1 through 7
    event_features = raw[:, 1:8].astype(np.float32)
    
    # Per-particle features
    particle_features = np.zeros(
        (n_events, max_particles, N_FEAT_PER_PARTICLE), dtype=np.float32
    )
    particle_mask = np.zeros((n_events, max_particles), dtype=np.float32)
    
    for i in range(n_events):
        n_obj = int(raw[i, 0])                             # Number of particles
        n_obj = min(n_obj, max_particles)                  # Cap at maximum
        
        for j in range(n_obj):
            col_start = OBJ_START + j * N_FEAT_PER_PARTICLE
            col_end = col_start + N_FEAT_PER_PARTICLE
            if col_end <= raw.shape[1]:
                particle_features[i, j, :] = raw[i, col_start:col_end]
                particle_mask[i, j] = 1.0                  # Mark as real particle
    
    return event_features, particle_features, particle_mask


# Parse both datasets
sig_evt, sig_parts, sig_mask = parse_data(sig_df)
bkg_evt, bkg_parts, bkg_mask = parse_data(bkg_df)

# Combine
event_features = np.concatenate([sig_evt, bkg_evt], axis=0)
particle_features = np.concatenate([sig_parts, bkg_parts], axis=0)
particle_mask = np.concatenate([sig_mask, bkg_mask], axis=0)
labels = np.concatenate([
    np.ones(len(sig_evt)),                                 # Signal → 1
    np.zeros(len(bkg_evt))                                 # Background → 0
]).astype(np.float32)

print(f"Total events: {len(labels)}")
print(f"Event features: {event_features.shape}")           # (N, 7)
print(f"Particle features: {particle_features.shape}")     # (N, 50, 4)
print(f"Signal: {labels.sum():.0f}, Background: {len(labels)-labels.sum():.0f}")

# Train/test split
(evt_train, evt_test,
 parts_train, parts_test,
 mask_train, mask_test,
 y_train, y_test) = train_test_split(
    event_features, particle_features, particle_mask, labels,
    test_size=0.2, random_state=42, stratify=labels
)

# Further split train → train/val
(evt_tr, evt_val,
 parts_tr, parts_val,
 mask_tr, mask_val,
 y_tr, y_val) = train_test_split(
    evt_train, parts_train, mask_train, y_train,
    test_size=0.15, random_state=42
)

print(f"Train: {len(y_tr)}, Val: {len(y_val)}, Test: {len(y_test)}")


In [ ]:

# ============================================================
# 2. DATASET AND DATALOADER
# ============================================================

class EventDataset(Dataset):
    """Custom dataset for event data."""
    def __init__(self, evt_feats, part_feats, mask, labels):
        self.evt = torch.FloatTensor(evt_feats)            # Event features
        self.parts = torch.FloatTensor(part_feats)         # Particle features
        self.mask = torch.FloatTensor(mask)                # Particle mask
        self.labels = torch.FloatTensor(labels).unsqueeze(1)  # Labels (N, 1)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.evt[idx], self.parts[idx], self.mask[idx], self.labels[idx]


BATCH_SIZE = 64

train_dataset = EventDataset(evt_tr, parts_tr, mask_tr, y_tr)
val_dataset = EventDataset(evt_val, parts_val, mask_val, y_val)
test_dataset = EventDataset(evt_test, parts_test, mask_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# ============================================================
# 3. JETVLAD MODEL
# ============================================================


class VLADPooling(nn.Module):
    """
    VLAD (Vector of Locally Aggregated Descriptors) pooling layer.
    
    Algorithm:
      1. For each particle i with feature z_i (dim D):
         - Compute soft assignment a_ik to each of K clusters
         - a_ik = softmax( w_k^T z_i + b_k )
      2. For each cluster k:
         - Accumulate weighted residuals: V_k = Σ_i a_ik * (z_i - c_k)
         - c_k is the cluster center
      3. Normalize each V_k (intra-normalization)
      4. Concatenate all V_k → flat descriptor of size K*D
      5. L2-normalize the full descriptor
    
    This produces a fixed-size, permutation-invariant representation
    that is richer than simple sum/mean pooling.
    
    Intuition:
      - Each cluster learns a "particle prototype" (e.g., hard particle, soft particle)
      - The residuals capture how particles in that category deviate from the prototype
      - Different signal/background events will have different residual patterns
    """
    def __init__(self, feature_dim, n_clusters=8):
        """
        Parameters:
        -----------
        feature_dim : int
            Dimension of input features per particle (D).
        n_clusters : int
            Number of VLAD clusters (K). More clusters = more expressive
            but more parameters. Typical: 4-16 for jet physics.
        """
        super(VLADPooling, self).__init__()
        
        self.feature_dim = feature_dim                     # D
        self.n_clusters = n_clusters                       # K
        
        # Soft assignment weights: linear projection for cluster assignment
        # Maps each particle feature z_i to K logits (one per cluster)
        self.assignment = nn.Sequential(
            nn.Linear(feature_dim, n_clusters),            # (D) → (K)
            # No softmax here — we apply it in forward() with masking
        )
        
        # Cluster centers: learnable parameters (K centers, each of dim D)
        # Initialized randomly; will be learned during training
        self.cluster_centers = nn.Parameter(
            torch.randn(n_clusters, feature_dim) * 0.1     # (K, D)
        )
    
    def forward(self, x, mask=None):
        """
        Parameters:
        -----------
        x : (batch, N, D) — per-particle features after encoding
        mask : (batch, N) — 1 for real particles, 0 for padding
        
        Returns:
        --------
        vlad : (batch, K * D) — VLAD descriptor (fixed size!)
        """
        batch_size, n_particles, feat_dim = x.shape
        
        # Step 1: Compute soft assignment scores
        # (batch, N, D) → (batch, N, K)
        assignment_logits = self.assignment(x)
        
        # Mask padded particles before softmax (set to -inf so softmax → 0)
        if mask is not None:
            # mask: (batch, N) → (batch, N, 1)
            mask_expanded = mask.unsqueeze(-1)             # For broadcasting
            # Set padded positions to very negative (softmax will give ~0)
            assignment_logits = assignment_logits + (1 - mask_expanded) * (-1e9)
        
        # Soft assignment via softmax over clusters (for each particle)
        # Each particle gets a probability distribution over K clusters
        soft_assign = F.softmax(assignment_logits, dim=-1)  # (batch, N, K)
        
        # Zero out padded particles explicitly
        if mask is not None:
            soft_assign = soft_assign * mask_expanded
        
        # Step 2: Compute residuals and aggregate
        # For each cluster k: V_k = Σ_i a_ik * (z_i - c_k)
        
        # Expand dimensions for broadcasting:
        # x: (batch, N, D) → (batch, N, 1, D)
        x_expanded = x.unsqueeze(2)                        # (batch, N, 1, D)
        
        # cluster_centers: (K, D) → (1, 1, K, D)
        centers = self.cluster_centers.unsqueeze(0).unsqueeze(0)  # (1, 1, K, D)
        
        # Residuals: z_i - c_k for all (i, k) pairs
        residuals = x_expanded - centers                   # (batch, N, K, D)
        
        # Weight residuals by soft assignment:
        # soft_assign: (batch, N, K) → (batch, N, K, 1)
        weights = soft_assign.unsqueeze(-1)                # (batch, N, K, 1)
        
        # Weighted residuals
        weighted_residuals = weights * residuals            # (batch, N, K, D)
        
        # Sum over particles: aggregate per cluster
        vlad = weighted_residuals.sum(dim=1)               # (batch, K, D)
        
        # Step 3: Intra-normalization (L2-normalize each cluster's descriptor)
        vlad = F.normalize(vlad, p=2, dim=-1)              # Normalize along D
        
        # Step 4: Flatten: (batch, K, D) → (batch, K*D)
        vlad = vlad.reshape(batch_size, -1)
        
        # Step 5: L2-normalize the full descriptor
        vlad = F.normalize(vlad, p=2, dim=-1)
        
        return vlad



In [ ]:

class JetVLAD(nn.Module):
    """
    JetVLAD: Full classification model using VLAD pooling.
    
    Architecture:
      1. Per-particle encoder (MLP): raw features → latent space
      2. VLAD pooling: variable-length particles → fixed-size descriptor
      3. Merge with event-level features
      4. Classifier head (MLP)
    
    The model is permutation-invariant (VLAD pooling is symmetric).
    """
    def __init__(self, particle_dims=4, event_dims=7, 
                 encoder_layers=[64, 128], n_clusters=8,
                 classifier_layers=[256, 128, 64]):
        """
        Parameters:
        -----------
        particle_dims : int
            Raw features per particle (pT, eta, phi, flag = 4).
        event_dims : int
            Number of event-level features (7).
        encoder_layers : list of int
            Hidden layers for per-particle encoder.
        n_clusters : int
            Number of VLAD clusters.
        classifier_layers : list of int
            Hidden layers for the classifier head.
        """
        super(JetVLAD, self).__init__()
        
        self.n_clusters = n_clusters
        
        # --- Per-particle encoder ---
        # Maps raw (pT, eta, phi, flag) to a learned representation
        enc_modules = []
        in_dim = particle_dims
        for out_dim in encoder_layers:
            enc_modules.append(nn.Linear(in_dim, out_dim))
            enc_modules.append(nn.BatchNorm1d(out_dim))    # Normalize
            enc_modules.append(nn.ReLU())                  # Non-linearity
            enc_modules.append(nn.Dropout(0.1))            # Light regularization
            in_dim = out_dim
        self.particle_encoder = nn.Sequential(*enc_modules)
        
        # Latent dimension = last encoder layer size
        latent_dim = encoder_layers[-1]
        
        # --- VLAD Pooling ---
        self.vlad_pool = VLADPooling(
            feature_dim=latent_dim,                        # D
            n_clusters=n_clusters                          # K
        )
        # Output size: K * D
        vlad_output_dim = n_clusters * latent_dim
        
        # --- Event feature encoder (small MLP) ---
        self.event_encoder = nn.Sequential(
            nn.Linear(event_dims, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU()
        )
        
        # --- Classifier head ---
        clf_modules = []
        in_dim = vlad_output_dim + 32                      # VLAD + event features
        for out_dim in classifier_layers:
            clf_modules.append(nn.Linear(in_dim, out_dim))
            clf_modules.append(nn.ReLU())
            clf_modules.append(nn.Dropout(0.2))
            in_dim = out_dim
        clf_modules.append(nn.Linear(in_dim, 1))           # Final output
        clf_modules.append(nn.Sigmoid())                   # Probability
        self.classifier = nn.Sequential(*clf_modules)
    
    def forward(self, event_feats, particle_feats, mask):
        """
        Parameters:
        -----------
        event_feats : (batch, 7) — event-level features
        particle_feats : (batch, N, 4) — per-particle (pT, eta, phi, flag)
        mask : (batch, N) — particle mask
        
        Returns:
        --------
        (batch, 1) — signal probability
        """
        batch_size, n_particles, _ = particle_feats.shape
        
        # --- Encode particles ---
        # Reshape for BatchNorm: (batch*N, 4) → encode → (batch*N, D)
        parts_flat = particle_feats.reshape(-1, particle_feats.size(-1))
        encoded_flat = self.particle_encoder(parts_flat)
        # Reshape back: (batch, N, D)
        latent_dim = encoded_flat.size(-1)
        encoded = encoded_flat.reshape(batch_size, n_particles, latent_dim)
        
        # Apply mask (zero out padded particles)
        encoded = encoded * mask.unsqueeze(-1)
        
        # --- VLAD pooling ---
        # (batch, N, D) → (batch, K*D)
        vlad_descriptor = self.vlad_pool(encoded, mask)
        
        # --- Encode event features ---
        event_encoded = self.event_encoder(event_feats)    # (batch, 32)
        
        # --- Classify ---
        combined = torch.cat([vlad_descriptor, event_encoded], dim=1)
        output = self.classifier(combined)
        
        return output



In [ ]:

# ============================================================
# 4. INSTANTIATE MODEL
# ============================================================

model = JetVLAD(
    particle_dims=N_FEAT_PER_PARTICLE,         # 4: pT, eta, phi, flag
    event_dims=7,                              # 7 event-level features
    encoder_layers=[64, 128],                  # Per-particle encoder
    n_clusters=8,                              # 8 VLAD clusters
    classifier_layers=[256, 128, 64]           # Classifier MLP
)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
print(f"\nJetVLAD Model:")
print(f"  Clusters (K): 8")
print(f"  Latent dim (D): 128")
print(f"  VLAD descriptor size: {8 * 128} = K × D")
print(f"  Total parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)



In [ ]:

# ============================================================
# 5. TRAINING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING JetVLAD")
print("=" * 60)

criterion = nn.BCELoss()                                   # Binary cross-entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)       # Adam optimizer
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5          # Halve LR after 5 stale epochs
)

EPOCHS = 60
best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 12
best_state = None

history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(EPOCHS):
    # --- Training phase ---
    model.train()
    train_loss = 0.0
    
    for evt, parts, mask, labels in train_loader:
        optimizer.zero_grad()                              # Clear gradients
        output = model(evt, parts, mask)                   # Forward pass
        loss = criterion(output, labels)                   # Compute loss
        loss.backward()                                    # Backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Clip gradients
        optimizer.step()                                   # Update weights
        train_loss += loss.item() * evt.size(0)            # Accumulate batch loss
    
    train_loss /= len(train_dataset)                       # Average over epoch
    history['train_loss'].append(train_loss)
    
    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels_list = []
    
    with torch.no_grad():                                  # No gradient computation
        for evt, parts, mask, labels in val_loader:
            output = model(evt, parts, mask)
            loss = criterion(output, labels)
            val_loss += loss.item() * evt.size(0)
            val_preds.append(output.numpy())
            val_labels_list.append(labels.numpy())
    
    val_loss /= len(val_dataset)
    val_preds = np.concatenate(val_preds).ravel()
    val_labels_arr = np.concatenate(val_labels_list).ravel()
    val_auc = roc_auc_score(val_labels_arr, val_preds)
    
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:3d}/{EPOCHS}: "
              f"train_loss={train_loss:.4f}, "
              f"val_loss={val_loss:.4f}, "
              f"val_auc={val_auc:.4f}, "
              f"lr={current_lr:.6f}")

# Restore best model
model.load_state_dict(best_state)
print(f"\nBest validation loss: {best_val_loss:.4f}")


In [ ]:
# ============================================================
# 6. EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("EVALUATION")
print("=" * 60)

model.eval()
all_preds = []
with torch.no_grad():
    for evt, parts, mask, labels in test_loader:
        output = model(evt, parts, mask)
        all_preds.append(output.numpy())

y_prob = np.concatenate(all_preds).ravel()                 # Predicted probabilities
y_pred = (y_prob >= 0.5).astype(int)                       # Hard predictions

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nJetVLAD Results:")
print(f"  Accuracy: {acc:.4f}")
print(f"  AUC-ROC:  {auc:.4f}")


In [ ]:

# ============================================================
# 7. ROC CURVE
# ============================================================

fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='purple', lw=2, label=f'JetVLAD (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate (Background Efficiency)')
plt.ylabel('True Positive Rate (Signal Efficiency)')
plt.title('ROC Curve — JetVLAD Classifier')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_jetvlad.pdf')
plt.show()

# Signal efficiency vs background rejection
plt.figure(figsize=(7, 6))
plt.plot(tpr, 1.0 / (fpr + 1e-10), color='purple', lw=2)  # 1/FPR = rejection
plt.xlabel('Signal Efficiency')
plt.ylabel('Background Rejection (1/FPR)')
plt.yscale('log')
plt.title('Signal Efficiency vs Background Rejection — JetVLAD')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sig_eff_bkg_rej_jetvlad.pdf')
plt.show()


In [ ]:

# ============================================================
# 8. TRAINING CURVES
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('JetVLAD Training Loss'); ax1.legend()

ax2.plot(history['val_auc'], label='Val AUC', color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('AUC')
ax2.set_title('JetVLAD Validation AUC'); ax2.legend()

plt.tight_layout(); plt.show()



In [ ]:
# ============================================================
# 9. INSPECT VLAD CLUSTERS
# ============================================================
# Visualize what each cluster has learned — examine cluster centers
# and soft assignments for signal vs background events

print("\n" + "=" * 60)
print("VLAD CLUSTER ANALYSIS")
print("=" * 60)

# Extract learned cluster centers
cluster_centers = model.vlad_pool.cluster_centers.detach().numpy()  # (K, D)
print(f"\nCluster centers shape: {cluster_centers.shape}")

# Compute soft assignments for a batch of signal and background events
model.eval()
with torch.no_grad():
    # Take first 200 signal and 200 background events from test set
    sig_mask_test = y_test == 1
    bkg_mask_test = y_test == 0
    
    n_show = min(200, sig_mask_test.sum(), bkg_mask_test.sum())
    
    # Get particle features and encode them
    sig_parts_sample = torch.FloatTensor(parts_test[sig_mask_test][:n_show])
    bkg_parts_sample = torch.FloatTensor(parts_test[bkg_mask_test][:n_show])
    sig_mask_sample = torch.FloatTensor(mask_test[sig_mask_test][:n_show])
    bkg_mask_sample = torch.FloatTensor(mask_test[bkg_mask_test][:n_show])
    
    # Encode particles through the encoder
    def get_assignments(parts, mask):
        """Get soft cluster assignments for particles."""
        batch_size, n_parts, _ = parts.shape
        parts_flat = parts.reshape(-1, parts.size(-1))
        encoded_flat = model.particle_encoder(parts_flat)
        latent_dim = encoded_flat.size(-1)
        encoded = encoded_flat.reshape(batch_size, n_parts, latent_dim)
        encoded = encoded * mask.unsqueeze(-1)
        
        # Get soft assignments from VLAD layer
        logits = model.vlad_pool.assignment(encoded)
        logits = logits + (1 - mask.unsqueeze(-1)) * (-1e9)
        assignments = F.softmax(logits, dim=-1)           # (batch, N, K)
        assignments = assignments * mask.unsqueeze(-1)
        return assignments
    
    sig_assignments = get_assignments(sig_parts_sample, sig_mask_sample)
    bkg_assignments = get_assignments(bkg_parts_sample, bkg_mask_sample)

# Average cluster usage: how much each cluster is used (summed assignment weight)
# Sum over particles, then average over events
sig_cluster_usage = sig_assignments.sum(dim=1).mean(dim=0).numpy()   # (K,)
bkg_cluster_usage = bkg_assignments.sum(dim=1).mean(dim=0).numpy()   # (K,)

# Plot cluster usage comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot: cluster usage
x_pos = np.arange(model.n_clusters)
width = 0.35
axes[0].bar(x_pos - width/2, sig_cluster_usage, width, label='Signal', color='blue', alpha=0.7)
axes[0].bar(x_pos + width/2, bkg_cluster_usage, width, label='Background', color='red', alpha=0.7)
axes[0].set_xlabel('Cluster Index (k)')
axes[0].set_ylabel('Average Assignment Weight')
axes[0].set_title('VLAD Cluster Usage: Signal vs Background')
axes[0].set_xticks(x_pos)
axes[0].legend()

# Heatmap: average assignment per cluster per pT bin
# This shows which clusters capture hard vs soft particles
sig_pts = sig_parts_sample[:, :, 0].numpy()                # pT values
sig_assign_np = sig_assignments.numpy()                    # (N_events, N_parts, K)

# Bin particles by pT and show average cluster assignment per bin
pt_bins = [0, 2, 5, 10, 20, 50, 100]
cluster_pt_profile = np.zeros((len(pt_bins)-1, model.n_clusters))

for b in range(len(pt_bins)-1):
    bin_mask = (sig_pts >= pt_bins[b]) & (sig_pts < pt_bins[b+1])
    if bin_mask.sum() > 0:
        # Average assignment for particles in this pT bin
        bin_assignments = sig_assign_np[bin_mask]          # (N_in_bin, K)
        cluster_pt_profile[b] = bin_assignments.mean(axis=0)

im = axes[1].imshow(cluster_pt_profile, aspect='auto', cmap='viridis')
axes[1].set_xlabel('Cluster Index (k)')
axes[1].set_ylabel('pT Bin')
axes[1].set_yticks(range(len(pt_bins)-1))
axes[1].set_yticklabels([f'{pt_bins[i]}-{pt_bins[i+1]}' for i in range(len(pt_bins)-1)])
axes[1].set_xticks(range(model.n_clusters))
axes[1].set_title('Cluster Assignment by pT Bin (Signal)')
plt.colorbar(im, ax=axes[1], label='Avg Assignment')

plt.tight_layout()
plt.savefig('vlad_cluster_analysis.pdf')
plt.show()

# Print cluster discrimination power
print("\nCluster discrimination (|sig_usage - bkg_usage| / total):")
diff = np.abs(sig_cluster_usage - bkg_cluster_usage)
total = sig_cluster_usage + bkg_cluster_usage
discrimination = diff / (total + 1e-8)
for k in range(model.n_clusters):
    bar = '█' * int(discrimination[k] * 50)
    print(f"  Cluster {k}: {discrimination[k]:.4f} {bar}")

# ============================================================
# 10. COMPARE WITH SIMPLE POOLING BASELINES
# ============================================================
# To show the benefit of VLAD over simpler approaches

print("\n" + "=" * 60)
print("ABLATION: VLAD vs SIMPLE POOLING")
print("=" * 60)



In [ ]:

class SimplePoolingBaseline(nn.Module):
    """Baseline model with sum/mean pooling instead of VLAD."""
    def __init__(self, particle_dims=4, event_dims=7,
                 encoder_layers=[64, 128], pooling='sum',
                 classifier_layers=[256, 128, 64]):
        super(SimplePoolingBaseline, self).__init__()
        
        self.pooling = pooling
        
        # Same encoder as JetVLAD
        enc_modules = []
        in_dim = particle_dims
        for out_dim in encoder_layers:
            enc_modules.append(nn.Linear(in_dim, out_dim))
            enc_modules.append(nn.BatchNorm1d(out_dim))
            enc_modules.append(nn.ReLU())
            enc_modules.append(nn.Dropout(0.1))
            in_dim = out_dim
        self.particle_encoder = nn.Sequential(*enc_modules)
        
        latent_dim = encoder_layers[-1]
        
        # Event encoder
        self.event_encoder = nn.Sequential(
            nn.Linear(event_dims, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU()
        )
        
        # Classifier
        clf_in = latent_dim + 32                           # pooled + event
        clf_modules = []
        for out_dim in classifier_layers:
            clf_modules.append(nn.Linear(clf_in, out_dim))
            clf_modules.append(nn.ReLU())
            clf_modules.append(nn.Dropout(0.2))
            clf_in = out_dim
        clf_modules.append(nn.Linear(clf_in, 1))
        clf_modules.append(nn.Sigmoid())
        self.classifier = nn.Sequential(*clf_modules)
    
    def forward(self, event_feats, particle_feats, mask):
        batch_size, n_parts, _ = particle_feats.shape
        
        # Encode
        parts_flat = particle_feats.reshape(-1, particle_feats.size(-1))
        encoded_flat = self.particle_encoder(parts_flat)
        latent_dim = encoded_flat.size(-1)
        encoded = encoded_flat.reshape(batch_size, n_parts, latent_dim)
        encoded = encoded * mask.unsqueeze(-1)
        
        # Pooling
        if self.pooling == 'sum':
            pooled = encoded.sum(dim=1)
        elif self.pooling == 'mean':
            n_real = mask.sum(dim=1, keepdim=True).clamp(min=1)
            pooled = encoded.sum(dim=1) / n_real
        elif self.pooling == 'max':
            encoded_masked = encoded + (1 - mask.unsqueeze(-1)) * (-1e9)
            pooled = encoded_masked.max(dim=1)[0]
        
        # Event + classify
        evt_enc = self.event_encoder(event_feats)
        combined = torch.cat([pooled, evt_enc], dim=1)
        return self.classifier(combined)


def quick_train_eval(model, name, epochs=40):
    """Quick train and evaluate helper."""
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_loss = float('inf')
    best_state = None
    
    for epoch in range(epochs):
        model.train()
        for evt, parts, mask, labels in train_loader:
            optimizer.zero_grad()
            out = model(evt, parts, mask)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for evt, parts, mask, labels in val_loader:
                out = model(evt, parts, mask)
                val_loss += criterion(out, labels).item() * evt.size(0)
        val_loss /= len(val_dataset)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    model.load_state_dict(best_state)
    model.eval()
    preds = []
    with torch.no_grad():
        for evt, parts, mask, labels in test_loader:
            preds.append(model(evt, parts, mask).numpy())
    preds = np.concatenate(preds).ravel()
    auc_val = roc_auc_score(y_test, preds)
    print(f"  {name:<20s}: AUC = {auc_val:.4f}")
    return preds, auc_val



In [ ]:

# Train baselines
print("\nTraining pooling baselines for comparison...")
_, auc_sum = quick_train_eval(
    SimplePoolingBaseline(pooling='sum'), "Sum Pooling"
)
_, auc_mean = quick_train_eval(
    SimplePoolingBaseline(pooling='mean'), "Mean Pooling"
)
_, auc_max = quick_train_eval(
    SimplePoolingBaseline(pooling='max'), "Max Pooling"
)

print(f"\n  {'JetVLAD (K=8)':<20s}: AUC = {auc:.4f}")
print(f"\n  Improvement over best simple pooling: "
      f"{(auc - max(auc_sum, auc_mean, auc_max))*100:.2f}% AUC")


In [ ]:

# ============================================================
# 11. VLAD CLUSTER COUNT COMPARISON
# ============================================================
# How does the number of clusters affect performance?

print("\n" + "=" * 60)
print("EFFECT OF NUMBER OF CLUSTERS (K)")
print("=" * 60)

cluster_aucs = {}
for K in [2, 4, 8, 12, 16]:
    print(f"\n  Training JetVLAD with K={K} clusters...")
    model_k = JetVLAD(
        particle_dims=N_FEAT_PER_PARTICLE,
        event_dims=7,
        encoder_layers=[64, 128],
        n_clusters=K,
        classifier_layers=[256, 128, 64]
    )
    _, auc_k = quick_train_eval(model_k, f"K={K}")
    cluster_aucs[K] = auc_k

# Plot
plt.figure(figsize=(8, 5))
plt.plot(list(cluster_aucs.keys()), list(cluster_aucs.values()), 'bo-', lw=2, markersize=8)
plt.axhline(y=max(auc_sum, auc_mean, auc_max), color='red', linestyle='--',
            label='Best simple pooling')
plt.xlabel('Number of VLAD Clusters (K)')
plt.ylabel('Test AUC')
plt.title('JetVLAD: Effect of Number of Clusters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vlad_n_clusters.pdf')
plt.show()